### Cell 1 Configuration

In [5]:
import os
import torch

class Config:
    # File Paths
    CSV_TRAIN = "train"
    IMG_DIR_TRAIN = "Image_data/train"
    
    # Model Settings
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4  
    
    # Hyperparameters
    EPOCHS = 3
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    VAL_SPLIT_RATIO = 0.2
    RANDOM_SEED = 42
    
    # Hardware Device Detection
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {Config.DEVICE}")
if Config.DEVICE.type == 'cuda':
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

Using Device: cuda
🔥 GPU Model: NVIDIA GeForce RTX 3050 6GB Laptop GPU


### CELL 2: DATASET & DATALOADERS

In [6]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

# 1. Define Training Transformations with Intensity Adjustments
train_transforms = transforms.Compose([
    transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    # Incorporating competition recommendation for brightness & contrast
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Testing requires original intensities without random jittering
val_transforms = transforms.Compose([
    transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Custom Dataset Definition
class IlluminationDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        
        # Determine if labels exist (Checks for train vs test layout)
        self.label_col = 'label' if 'label' in self.df.columns else ('Label' if 'Label' in self.df.columns else None)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        if not img_id.endswith('.png'):
            img_id = f"{img_id}.png"
            
        img_path = os.path.join(self.img_dir, img_id)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        if self.label_col is not None:
            label = int(self.df.iloc[idx][self.label_col])
            return image, label
        else:
            return image, img_id

# 3. Stratified Splitting and Dataloader Initialization
# Read primary metadata sheet
df_metadata = pd.read_csv(Config.CSV_TRAIN)
label_col_name = 'label' if 'label' in df_metadata.columns else 'Label'

# Perform split guaranteeing proportional class balance
train_df, val_df = train_test_split(
    df_metadata, 
    test_size=Config.VAL_SPLIT_RATIO, 
    stratify=df_metadata[label_col_name], 
    random_state=Config.RANDOM_SEED
)

# Instantiate PyTorch Datasets
train_dataset = IlluminationDataset(train_df, Config.IMG_DIR_TRAIN, transform=train_transforms)
val_dataset = IlluminationDataset(val_df, Config.IMG_DIR_TRAIN, transform=val_transforms)

# Prepare Dataloaders utilizing local GPU optimizations
train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=Config.NUM_WORKERS, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS, 
    pin_memory=True
)

print(f"✅ Data Preparation Complete: {len(train_dataset)} Train samples, {len(val_dataset)} Validation samples.")

✅ Data Preparation Complete: 1200 Train samples, 300 Validation samples.


### CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

In [7]:
import torch.nn as nn
import torchvision.models as models

def build_illumination_model():
    # 1. Load the pre-trained ResNet18 model
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    # 2. Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False

    # 3. Unfreeze Layer 4 (The final convolutional block)
    # This allows the network to learn domain-specific illumination features
    for param in model.layer4.parameters():
        param.requires_grad = True

    # 4. Replace the Fully Connected (fc) classification head
    # Newly created layers automatically have requires_grad = True
    num_features = model.fc.in_features
    
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),            # Regularization to prevent overfitting
        nn.Linear(num_features, 3)    # 3 Output nodes: Dark(0), Normal(1), Bright(2)
    )

    return model

# Instantiate the model and move it to the GPU/CPU
model = build_illumination_model().to(Config.DEVICE)

# Verify the freezing strategy worked by counting parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✅ Model Initialized on {Config.DEVICE}!")
print(f"🔒 Total Parameters: {total_params:,}")
print(f"🔥 Trainable Parameters: {trainable_params:,} (Layer 4 + FC Head)")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\kv997/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:38<00:00, 1.23MB/s]


✅ Model Initialized on cuda!
🔒 Total Parameters: 11,178,051
🔥 Trainable Parameters: 8,395,267 (Layer 4 + FC Head)


#### Training on 8 million parameters